In [ ]:
Sys.time()

In [ ]:
suppressPackageStartupMessages(library(ArchR))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(RColorBrewer))
suppressPackageStartupMessages(library(cowplot))
suppressPackageStartupMessages(library(gridExtra))
suppressPackageStartupMessages(library(ggsci))
suppressPackageStartupMessages(library(scales))
suppressPackageStartupMessages(library(viridis))
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(BSgenome))
suppressPackageStartupMessages(library(BSgenome.Hsapiens.UCSC.hg38))

In [ ]:
projdir <- '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'

In [ ]:
rds_path <- paste0(projdir, 'atac/peaks/archr/clusters/start_fail1.differential.rds')
peaks_rds <- readRDS(rds_path)

In [ ]:
assays(peaks_rds)

In [ ]:
peaks_df <- as.data.frame(rowData(peaks_rds))
head(peaks_df)

In [ ]:
rownames(peaks_df) <- paste0(peaks_df$seqnames, '_', peaks_df$start)

In [ ]:
fc <- assays(peaks_rds)$Log2FC
fdr <- assays(peaks_rds)$FDR
rownames(fc) <- rownames(peaks_df)
rownames(fdr) <- rownames(peaks_df)
dim(fc)
dim(fdr)

In [ ]:
thresh <- 0.01
mask <- fdr$Start < thresh
start_fc <- fc[mask,]
start_fc <- start_fc[!is.na(start_fc$Fail1) | !is.na(start_fc$Start),]
start_fdr <- fdr[mask,]
start_fdr <- start_fdr[!is.na(start_fdr$Fail1) | !is.na(start_fdr$Start),]
dim(start_fc)

In [ ]:
thresh <- 0.01
mask <- fdr$Fail1 < thresh
fail1_fc <- fc[mask,]
fail1_fc <- fail1_fc[!is.na(fail1_fc$Fail1) | !is.na(fail1_fc$Start),]
fail1_fdr <- fdr[mask,]
fail1_fdr <- fail1_fdr[!is.na(fail1_fdr$Fail1) | !is.na(fail1_fdr$Start),]
dim(fail1_fc)

In [ ]:
head(start_fc)

In [ ]:
head(fail1_fc)

In [ ]:
mask <- rownames(peaks_df) %in% rownames(start_fc)
start_peaks <- peaks_df[mask,]

mask <- rownames(peaks_df) %in% rownames(fail1_fc)
fail1_peaks <- peaks_df[mask,]

dim(start_peaks)
dim(fail1_peaks)

In [ ]:
final_start <- data.frame(chrom=start_peaks$seqnames, start=start_peaks$start, end=start_peaks$end,
            foldchange=start_fc$Start, fdr=start_fdr$Start)

final_fail1 <- data.frame(chrom=fail1_peaks$seqnames, start=fail1_peaks$start, end=fail1_peaks$end,
            foldchange=fail1_fc$Fail1, fdr=fail1_fdr$Fail1)

dim(final_start)
dim(final_fail1)

In [ ]:
head(final_start)

In [ ]:
mask <- final_start$foldchange > 0
tmp_start <- final_start[mask,]
dim(tmp_start)

In [ ]:
mask <- final_fail1$foldchange > 0
tmp_fail1 <- final_fail1[mask,]
dim(tmp_fail1)

In [ ]:
out <- paste0(projdir, '/csv/atac/peaks/chromhmm/start_fail1/Start.csv')
fwrite(tmp_start, out, sep='\t')

out <- paste0(projdir, '/csv/atac/peaks/chromhmm/start_fail1/Fail1.csv')
fwrite(tmp_fail1, out, sep='\t')

In [ ]:
out <- paste0(projdir, '/bed/peaks/chromhmm/start_fail1/Start.bed')
fwrite(tmp_start, out, sep='\t', col.names=FALSE)

out <- paste0(projdir, '/bed/peaks/chromhmm/start_fail1/Fail1.bed')
fwrite(tmp_fail1, out, sep='\t', col.names=FALSE)

In [ ]:
sessionInfo()

In [ ]:
print('Roll call! Which files are in here?')
for(f in list.files('.')){
    print(f)
    print('Here!')
}

In [ ]:
Sys.time()